In [1]:
import sys
sys.path.append('..')

import os
from loader import read_sentences, read_dictionary
from spelling_checker.candidates import BKTree, SymSpell
from spelling_checker.ngram_model import KNgramModel
from tqdm import tqdm
import random


In [2]:
max_distance = 1

language = "es"
misfit_file = True

data_dir = "./data"

dict_path = os.path.join(data_dir, "levidromelist-dicts", "spanish.txt")

hunspell_dir = os.path.join(data_dir, "hunspell")
unmunched_path = os.path.join(hunspell_dir, "es_ES_unmunched_words.txt")

sentences_path = os.path.join(data_dir, f"{language}_sentences.txt")
train_sentences_path = os.path.join(data_dir, f"{language}_train_sentences.txt")

model_dir = os.path.join("./models", f"distance_{max_distance}")
os.makedirs(model_dir, exist_ok=True)

bk_tree_path = os.path.join(model_dir, "bk_tree.pkl")
sym_spell_path = os.path.join(model_dir, "sym_spell.pkl")

forward_lm_path = os.path.join(model_dir, "forward_lm.pkl")
backward_lm_path = os.path.join(model_dir, "backward_lm.pkl")


In [3]:
dictionary_words = read_dictionary(unmunched_path)
print(len(dictionary_words))


649607


In [4]:
sentences = read_sentences(train_sentences_path)


In [5]:
print(f"{len(sentences):,}")


535,269


In [6]:
shuffled_words = list(dictionary_words)
random.shuffle(shuffled_words)


In [7]:
sym_spell = SymSpell(max_dist=max_distance)

for word in tqdm(shuffled_words, desc="Building SymSpell"):
	sym_spell.add(word)


Building SymSpell: 100%|██████████| 649607/649607 [00:08<00:00, 76781.53it/s]


In [8]:
sym_spell.save(sym_spell_path)


In [9]:
# tree = BKTree(max_dist=max_distance)

# for word in tqdm(dictionary_words, desc="Building BKTree"):
# 	tree.add(word)


In [10]:
# tree.save(bk_tree_path)


In [11]:
order = 2
discount = 0.75
unk_threshold = 1


In [12]:
forward_lm = KNgramModel(order=order, discount=discount, unk_threshold=unk_threshold)
forward_lm.train(sentences)


Training: 100%|██████████| 535269/535269 [00:07<00:00, 69515.45it/s]


In [13]:
forward_lm.save(forward_lm_path)


In [14]:
reversed_sentences = [list(reversed(sent)) for sent in sentences]

backward_lm = KNgramModel(order=order, discount=discount, unk_threshold=unk_threshold)
backward_lm.train(reversed_sentences)


Training: 100%|██████████| 535269/535269 [00:08<00:00, 63748.40it/s]


In [15]:
backward_lm.save(backward_lm_path)
